In [1]:
# PDF Loader
from langchain_community.document_loaders import PyPDFDirectoryLoader

# Text Splitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embeddings
from langchain_huggingface import HuggingFaceEmbeddings

# Vector Store
from langchain_community.vectorstores import FAISS

# LLM - Ollama
from langchain_ollama import ChatOllama

# Prompt Template
from langchain_classic.prompts import PromptTemplate

# Retrieval QA Chain
from langchain_classic.chains import RetrievalQA

# Gradio
import gradio as gr

C:\Users\user\AppData\Local\Temp\ipykernel_11840\4003142043.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFDirectoryLoader


In [3]:
# Load PDF Documents

loader = PyPDFDirectoryLoader("../data/IT_Resumes/")

docs_before_split = loader.load()

print("Number of pages:", len(docs_before_split))

Number of pages: 42


In [4]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=50
)

docs_after_split = text_splitter.split_documents(docs_before_split)

print("Number of chunks:", len(docs_after_split))

Number of chunks: 240


In [5]:
huggingface_embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [6]:
vectorstore = FAISS.from_documents(
    docs_after_split,
    huggingface_embeddings
)

print("Vector database created successfully!")
print("Number of vectors:", vectorstore.index.ntotal)

Vector database created successfully!
Number of vectors: 240


In [7]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

print("Retriever created successfully!")

Retriever created successfully!


In [8]:
query = "Find candidates with TensorFlow and AWS experience"

relevant_documents = retriever.invoke(query)

print("Number of retrieved chunks:", len(relevant_documents))

for i, doc in enumerate(relevant_documents):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content[:500])

Number of retrieved chunks: 3

--- Result 1 ---
*Superior knowledge of INFOSEC/NETOPS
techniques to determine requirements for information technology equipment and systems. Excellent knowledge and ability to negotiate in financial
control, budget systems, and security management. Working knowledge of budget and cost analysis processes. *Thorough knowledge of Office of
Management and Budget (OMB), Department of Defense (DoD) and U.S. Air Force regulations, principles and practices pertaining to logistics,
INFOSEC/NETOPS, installations, and mis

--- Result 2 ---
Excellent Verbal and written communication
Agile, Functional, Project leadership,Written communication
AI, Information technology, Project Management
Budgeting, Insurance, Python
Budgets, ITIL, Quality
Budget, Leadership, Research
Business intelligence, Marketing, Risk assessment
Business process re-engineering, Access, Sales
Change management, Excel, Scheduling
Conflict resolution, Migration, SDLC
Critical thinking, Network desi

In [9]:

model = ChatOllama(
    model="llama3.2:3b",
    temperature=0.3
)

In [10]:
template = """
You are an AI assistant specialized in screening resumes.

Use the following resume information to answer the question.

Context:
{context}

Question:
{question}

Instructions:
- Identify candidates based only on the provided context.
- Do not invent candidate information.
- If the information is not available, say so.
- Give a clear and concise answer.

Answer:
"""

PROMPT = PromptTemplate(
    template=template,
    input_variables=["context", "question"]
)

print("Prompt template created successfully!")

Prompt template created successfully!


In [11]:
retrievalQA = RetrievalQA.from_chain_type(
    llm=model,
    retriever=retriever,
    return_source_documents=True,
    chain_type="stuff",
    chain_type_kwargs={"prompt": PROMPT}
)

print("RetrievalQA chain created successfully!")

RetrievalQA chain created successfully!


In [12]:
print("Vectorstore:", vectorstore is not None)
print("Retriever:", retriever is not None)
print("Prompt:", PROMPT is not None)
print("LLM:", model is not None)

Vectorstore: True
Retriever: True
Prompt: True
LLM: True


In [13]:
query = "Find candidates with TensorFlow + AWS experience"

result = retrievalQA.invoke({"query": query})

print(result["result"])

Based on the provided resume context, I found no mention of TensorFlow or AWS experience. The resume only lists general skills and certifications, but does not specifically mention TensorFlow or AWS.


In [14]:
query = "Find candidates with SQL experience"

result = retrievalQA.invoke({"query": query})

print(result["result"])

Based on the provided context, the following candidates have SQL experience:

1. Database Programmer/Analyst (.NET Developer) - has experience with SQL, Oracle SQL, and SQL Server.
2. DIRECTOR OF INFORMATION TECHNOLOGY - has experience with Oracle SQL, SQL Server, and PL/SQL.
3. Database Programmer/Analyst (.NET Developer) - has experience with ADO.NET data, which includes SQL.

Note that the DIRECTOR OF INFORMATION TECHNOLOGY also has experience with SQL*Plus, but it's not explicitly stated that they have hands-on experience with it.


In [17]:
def screen_resume(question):

    result = retrievalQA.invoke({"query": question})

    answer = result["result"]

    sources = []

    for doc in result["source_documents"]:
        file_name = doc.metadata.get("source", "Unknown")
        page = doc.metadata.get("page", "Unknown")

        source_info = f"{file_name} - Page {page}"

        if source_info not in sources:
            sources.append(source_info)

    sources_text = "\n".join(sources)

    return answer, sources_text

In [18]:
demo = gr.Interface(
    fn=screen_resume,

    inputs=gr.Textbox(
        lines=2,
        label="Your Question",
        placeholder="Ask about candidates' skills and experience..."
    ),

    outputs=[
        gr.Textbox(
            label="Screening Result",
            lines=10
        ),
        gr.Textbox(
            label="Sources",
            lines=4
        )
    ],

    title="AI Resume Screener",
    description="An AI-powered tool to screen resumes and identify candidates based on their skills and experience."
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
